# Natural Language Processing



This project will give you practical experience using Natural Language Processing techniques. This project is in three parts:
- in part 1) you will use a dataset in a CSV file
- in part 2) you will use the Wikipedia API to directly access content
on Wikipedia.
- in part 3) you will make your notebook interactive


### Part 1)



- The CSV file is available at https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv
- The file contains a list of famous people and a brief overview.
- The goal of part 1) is to ...
  1. Pick one person from the list ( the reference person ) and output 10 other people who's overview are "closest" to the reference person in a Natural Language Processing sense
  1. Also output the sentiment of the overview of the reference person



### Part 2)



- For the same reference person that you chose in Part 1), use the Wikipedia API to access the whole content of the reference person's Wikipedia page.
- The goal of Part 2) is to ...
  1. Print out the text of the Wikipedia article for the reference person
  1. Determine the sentiment of the text of the Wikipedia page for the reference person
  1. Collect the text of the Wikipedia pages from the 10 nearest neighbors from Part 1)
  1. Determine the nearness ranking of these 10 people to your reference person based on their entire Wikipedia page
  1. Compare, i.e. plot,  the nearest ranking from Step 1) with the Wikipedia page nearness ranking.  A difference of the rank is one means of comparison.



### Part 3)


Make an interactive notebook where a user can choose or enter a name and the notebook displays the 10 closest individuals.

In addition to presenting the project slides, at the end of the presentation each student will demonstrate their code using a famous person suggested by the other students that exists in the DBpedia set.


## Data Collection

### Import

In [ ]:
import duckdb
import pandas as pd
import os
import numpy as np
from google.colab import userdata


# TextBlob: cleans text (singularize) and scores sentiment
from textblob import TextBlob, Word

# TfidfVectorizer: turns each overview into a row of numbers (one column per word)
from sklearn.feature_extraction.text import TfidfVectorizer

# NearestNeighbors: finds the rows (people) closest to a reference row
from sklearn.neighbors import NearestNeighbors



## Convert CSV to Parquet

In [ ]:
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'
url

'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'

In [ ]:
# File name without the extension: 'NLP'
base_file = url.split("/")[-1].split(".")[0]
base_file

'NLP'

In [ ]:
# Read the remote CSV and write it straight to a local parquet file (lesson 102's ETL step)
# No all_varchar here: DuckDB auto-detects types, so number columns stay numbers
con = duckdb.connect()
con.execute(f"""
    COPY (
        SELECT *
        FROM read_csv('{url}')
    )
    TO '{base_file}.parquet'
    (FORMAT PARQUET);
""")

! ls -l {base_file}.parquet

-rw-r--r-- 1 root root 49460246 Nov 27 16:25 NLP.parquet


### Push to Hugging Face

In [ ]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account

'hmsealy'

In [ ]:
hf_repo = os.environ["HF_REPO"] = "Data_Science-22"
hf_repo

'Data_Science-22'

In [ ]:
hf_file = os.environ["HF_FILE"] = f"{base_file}.parquet"
hf_file

'NLP.parquet'

In [ ]:
# Get the token from Colab Secrets. Only shows the first 3 and last 4 characters
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"

'hf_ ... kJTa'

In [ ]:
! hf auth login --token ${HF_TOKEN}

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"

In [ ]:
print(hf_upload.stdout)

✓ Uploaded
  url: https://huggingface.co/datasets/hmsealy/Data_Science-22/commit/53b83f43a5d0a777ceb6ff484e323f69b898e28f



### Download TextBlob word lists

In [ ]:
%%capture
# %%capture hides the long download log
!python -m textblob.download_corpora

## Load Data

### Load first 100 rows

In [ ]:
# Pass 04 rule: use only the first 100 rows of the CSV
df = duckdb.query(f'select * from "{base_file}.parquet" limit 100').df()
df.shape

(100, 3)

## Data checks

In [ ]:
import requests                      # fetch web pages (HTML lecture, 315)
from urllib.parse import unquote     # turns %C3%A9 back into é

### Check 1: Missing names

In [ ]:
# How many rows have no name at all? We want 0.
df["name"].isna().sum()

np.int64(0)

### Check 2: Duplicate names

In [ ]:
# keep=False shows EVERY copy of a repeated name, not just the second one
df[df["name"].duplicated(keep=False)][["name"]]

,name


### Check 3: Names with parentheses

In [ ]:
# Wikipedia adds (job) to tell apart people with the same name, e.g. "Paddy Dunne (Gaelic footballer)"
# regex=False means "(" is just a normal character here
df[df["name"].str.contains("(", regex=False)][["name"]]

,name
17,Paddy Dunne (Gaelic footballer)
27,Ian Mitchell (author)
31,Ted Hill (mathematician)
41,Bob Adams (American football)
50,Professor Lyrical (rapper)
53,Vincent Fang (lyricist)
66,Elizabeth Gunn (author)
74,Mike Powell (journalist)
77,John Warner (writer)
89,Sam Leach (artist)


### Check 4: Names with commas

In [ ]:
# Titles like "Anthony Gueterbock, 18th Baron Berkeley"
df[df["name"].str.contains(",", regex=False)][["name"]]

,name
13,"Anthony Gueterbock, 18th Baron Berkeley"


### Check 5: Names with unusual characters

In [ ]:
# Anything that is NOT a plain letter, space, period, apostrophe or hyphen.
# This catches digits, accents (é, ö), parentheses and commas.
df[df["name"].str.contains(r"[^A-Za-z .'\-]")][["name"]]

,name
13,"Anthony Gueterbock, 18th Baron Berkeley"
17,Paddy Dunne (Gaelic footballer)
27,Ian Mitchell (author)
31,Ted Hill (mathematician)
35,Freimut B%C3%B6rngen
36,Th%C3%BCring Br%C3%A4m
41,Bob Adams (American football)
50,Professor Lyrical (rapper)
53,Vincent Fang (lyricist)
66,Elizabeth Gunn (author)


### Check 6: Name not found in its own overview

In [ ]:
# Is the first word of the name in the overview text?
# If not, the person may be known by a stage name (e.g. G-Enka = Henry Kõrvits)
# or an accent was removed in the text (é -> e)
first_words = [n.lower().split()[0] for n in df["name"]]
df["name_in_text"] = [w in t for w, t in zip(first_words, df["text"])]

# ~ means "NOT", so this shows rows where the name was NOT found
df[~df["name_in_text"]][["name", "text"]]

,name,text
4,G-Enka,henry krvits born 30 december 1974 in tallinn ...
33,Bob Reece,robert scott reece born january 5 1951 is an a...
36,Th%C3%BCring Br%C3%A4m,thring brm born 10 april 1944 is a swiss compo...
72,Ulf-Dietrich Reips,prof dr ulfdietrich reips is a full professor ...
75,Vladimir Yurchenko,uladzimir vasilyevich yurchanka belarusian rus...
86,John-Paul Himka,johnpaul himka born may 18 1949 in detroit mic...


### Check 7: Page title from each URI

In [ ]:
# <http://dbpedia.org/resource/Jonathan_Hoefler>  ->  Jonathan_Hoefler
df["title"] = (
    df["URI"]
    .str.replace("<http://dbpedia.org/resource/", "")
    .str.replace(">", "")
    .apply(unquote)
)

# Does the title match the name? (title uses _ instead of spaces)
df[df["title"].str.replace("_", " ") != df["name"]][["name", "title"]]

,name,title
35,Freimut B%C3%B6rngen,Freimut_Börngen
36,Th%C3%BCring Br%C3%A4m,Thüring_Bräm
73,Marcel J. Melan%C3%A7on,Marcel_J._Melançon
85,Zvonimir Juri%C4%87,Zvonimir_Jurić


### Fix coded names

In [ ]:
# Some names are stored as web-address codes: "B%C3%B6rngen" should be "Börngen"
# unquote() turns them back into normal letters. Names without codes don't change.
df["name"] = df["name"].apply(unquote)

# Show the 4 fixed names
df.loc[[35, 36, 73, 85], "name"]

,name
35,Freimut Börngen
36,Thüring Bräm
73,Marcel J. Melançon
85,Zvonimir Jurić


### Check 8: Is each person on Wikipedia, with a real article?

In [ ]:
# Wikipedia asks programs to identify themselves (like user_agent in Pass 02)
headers = {"User-Agent": "neefmister"}

def check_wiki_page(title):
    url = "https://en.wikipedia.org/wiki/" + title
    page = requests.get(url, headers=headers)      # same as the HTML lecture

    if page.status_code != 200:                    # 200 = OK, 404 = page not found
        status = f"error {page.status_code}"
    elif 'id="disambigbox"' in page.text or "may refer to" in page.text:
        status = "disambiguation"                  # a list of different people with this name
    else:
        status = "ok"

    return status

# Check all 100 people. This takes about a minute (100 web requests).
df["wiki_status"] = df["title"].apply(check_wiki_page)

### Summary of Wikipedia checks

In [ ]:
# How many of each result?
df["wiki_status"].value_counts()

,count
wiki_status,
ok,90
error 404,9
disambiguation,1


### People whose link is NOT ok

In [ ]:
df[df["wiki_status"] != "ok"][["name", "title", "wiki_status"]]

,name,title,wiki_status
8,Grant Nelson,Grant_Nelson,disambiguation
15,Joerg Steineck,Joerg_Steineck,error 404
26,Anthony Caruana,Anthony_Caruana,error 404
27,Ian Mitchell (author),Ian_Mitchell_(author),error 404
37,Aharon Solomons,Aharon_Solomons,error 404
38,Steven Weil,Steven_Weil,error 404
46,Will Tuttle,Will_Tuttle,error 404
59,Alexis Bhagat,Alexis_Bhagat,error 404
73,Marcel J. Melançon,Marcel_J._Melançon,error 404
93,Martin Iveson,Martin_Iveson,error 404


### Findings (first 100 rows)

**Names**
- Missing names: 0
- Duplicate names: 0
- Names with a job in parentheses: 12, e.g. "Paddy Dunne (Gaelic footballer)". Wikipedia uses this
  to tell apart people with the same name.
- Names with a comma: 1, "Anthony Gueterbock, 18th Baron Berkeley" (a title)
- Names stored as web-address codes: 4, e.g. "Freimut B%C3%B6rngen" -> fixed to "Freimut Börngen" with unquote()
- Name not found in its own overview: 6
  - Stage name or nickname: G-Enka (Henry Kõrvits), Bob Reece (Robert)
  - Different spelling: Vladimir Yurchenko (Uladzimir Yurchanka)
  - Accents removed from the overview text: Thüring Bräm ("thring brm")
  - Hyphen removed from the overview text: Ulf-Dietrich Reips, John-Paul Himka (not real problems,
    the check can't match "ulf-dietrich" to "ulfdietrich")

**Wikipedia links**
- 90 ok, 9 not found (404), 1 disambiguation page
- Not found: Joerg Steineck, Anthony Caruana, Ian Mitchell (author), Aharon Solomons, Steven Weil,
  Will Tuttle, Alexis Bhagat, Marcel J. Melançon, Martin Iveson
- Disambiguation: Grant Nelson
- 10% of the first 100 links don't lead to a usable article

## Part 1

### Pick the reference person

In [ ]:
ref_name = "Jonathan Hoefler"
ref_index = 12   # Jonathan's row number, found by looking at df (same as with 20 rows)
ref_index

12

### Clean text (singularize)

In [ ]:
# Turn each word into its singular form so "typefaces" and "typeface" count as the same word
def singularize_text(text):
    words = text.split()                               # split the overview into a list of words
    singles = [Word(w).singularize() for w in words]   # singularize each word
    return " ".join(singles)                           # glue the words back into one string

# Apply the function to every overview and keep the result in a new column
df["text_clean"] = df["text"].apply(singularize_text)

# Compare the original and cleaned text for the reference person
print(df.loc[ref_index, "text"][:200])
print(df.loc[ref_index, "text_clean"][:200])

jonathan hoefler born august 22 1970 is an american typeface designer hoefler pronounced heffler founded the hoefler type foundry in 1989 a type foundry in new york in 1999 hoefler began working with 
jonathan hoefler born august 22 1970 is an american typeface designer hoefler pronounced heffler founded the hoefler type foundry in 1989 a type foundry in new york in 1999 hoefler began working with 


In [ ]:
# stop_words='english' drops filler words like "the", "is", "and"
tfidf_vec = TfidfVectorizer(stop_words="english")

# fit_transform learns the vocabulary and builds the matrix:
# 100 rows (people) x one column per unique word
tf_idf = tfidf_vec.fit_transform(df["text_clean"])
tf_idf.shape

(100, 6194)

### Teach NearestNeighbors

In [ ]:
# Teach NearestNeighbors where all 100 people sit in "word space"
nn = NearestNeighbors().fit(tf_idf)

# Ask for 11 neighbors, not 10: the closest match is always Jonathan himself (distance 0)
distances, indices = nn.kneighbors(
    X = tf_idf[ref_index],
    n_neighbors = 11,
)
distances, indices

(array([[0.        , 1.36415423, 1.36928942, 1.37562054, 1.38161353,
         1.38164241, 1.38182967, 1.38292405, 1.38313601, 1.38351166,
         1.38451025]]),
 array([[12, 58,  5, 96, 43, 74, 37, 26, 19, 89, 38]]))

In [ ]:
# Pull out the people at those row numbers, and add their distances
neighbors = df.iloc[indices[0]][["name"]].copy()
neighbors["distance"] = distances[0]

# Drop the first row (Jonathan himself) and keep only the 10 closest
neighbors = neighbors.iloc[1:11]

# Add a rank column: 1 = closest
neighbors.insert(0, "rank", range(1, 11))
neighbors.reset_index(drop=True)

,rank,name,distance
0,1,Christopher Sorrentino,1.364154
1,2,Sam Henderson,1.369289
2,3,Armen Ra,1.375621
3,4,Rob Sheffield,1.381614
4,5,Mike Powell (journalist),1.381642
5,6,Aharon Solomons,1.381830
6,7,Anthony Caruana,1.382924
7,8,John Angus Campbell,1.383136
8,9,Sam Leach (artist),1.383512
9,10,Steven Weil,1.384510


### Sentiment

In [ ]:
# Use the ORIGINAL text for sentiment, not the cleaned text
ref_blob = TextBlob(df.loc[ref_index, "text"])

# polarity: -1 (negative) to +1 (positive)
# subjectivity: 0 (factual) to 1 (opinion)
ref_blob.sentiment

Sentiment(polarity=0.19034090909090906, subjectivity=0.37683080808080804)